# Task 2: Apply Perturbations to ALS Disease-Specific Genes

Now we apply the workflow to actual ALS genes, perturbing both disease and healthy cells.

**Gene selection**: Two sources. First, the canonical ALS genes from the literature (SOD1, TARDBP, FUS, C9orf72). Second, genes identified in the Pineda et al. dataset itself (POU3F1, SCN4B, STMN2); these are specific to the motor cortex vulnerability patterns in this study. Since this is a sporadic ALS cohort (no C9orf72 expansion carriers), the pathology is TDP-43 driven; TARDBP and its downstream targets are the most relevant.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse
from tqdm import tqdm
import torch
import pickle
import os

sns.set_theme(style='whitegrid')

from helical.models.geneformer import Geneformer, GeneformerConfig

## 1. Load data and baseline embeddings from Task 1

In [ ]:
DATA_PATH = "../data/counts_combined_filtered_BA4_sALS_PN.h5ad"
adata = sc.read_h5ad(DATA_PATH)

# Load cached baseline embeddings
embeddings_baseline = np.load('../data/embeddings_baseline.npy')
adata.obsm['X_geneformer'] = embeddings_baseline

print(f"Dataset: {adata.shape[0]} cells x {adata.shape[1]} genes")
print(f"Baseline embeddings: {embeddings_baseline.shape}")

In [ ]:
# Identify key metadata columns
# Adjust these based on Task 1 EDA results
print("Available columns:", list(adata.obs.columns))
print()

# Auto-detect disease and cell type columns
DISEASE_COL = None
CELLTYPE_COL = None

for col in adata.obs.columns:
    col_lower = col.lower()
    if DISEASE_COL is None and any(kw in col_lower for kw in ['disease', 'condition', 'diagnosis', 'status', 'group']):
        DISEASE_COL = col
    if CELLTYPE_COL is None and any(kw in col_lower for kw in ['cell_type', 'subclass', 'celltype', 'cluster', 'type']):
        CELLTYPE_COL = col

print(f"Disease column: {DISEASE_COL}")
if DISEASE_COL:
    print(adata.obs[DISEASE_COL].value_counts())
print(f"\nCell type column: {CELLTYPE_COL}")
if CELLTYPE_COL:
    print(adata.obs[CELLTYPE_COL].value_counts())

In [ ]:
# Column names from data exploration (Task 1)
# PN = post-mortem normal (healthy controls), ALS = sporadic ALS (disease)
DISEASE_COL   = 'Condition'
CELLTYPE_COL  = 'CellType'
SUBTYPE_COL   = 'SubType'
HEALTHY_LABEL = 'PN'
DISEASE_LABEL = 'ALS'

healthy_mask = (adata.obs[DISEASE_COL] == HEALTHY_LABEL).values
disease_mask = (adata.obs[DISEASE_COL] == DISEASE_LABEL).values

print(f"Condition column: {DISEASE_COL}")
print(f"  Healthy ({HEALTHY_LABEL}): {healthy_mask.sum():,} cells")
print(f"  Disease ({DISEASE_LABEL}): {disease_mask.sum():,} cells")

## 2. Select ALS disease genes

In [ ]:
# Candidate ALS genes, ordered by relevance
ALS_GENES_CANDIDATES = [
    # Core ALS genetic drivers
    'TARDBP',   # TDP-43: proteinopathy in 97% of ALS, nuclear-to-cytoplasmic mislocalization
    'SOD1',     # First ALS gene discovered, causes protein aggregation
    'FUS',      # RNA-binding protein, cytoplasmic aggregation
    'C9orf72',  # Most common genetic cause, hexanucleotide repeat expansion
    
    # Downstream effectors / therapeutic targets
    'STMN2',    # Stathmin-2, downstream of TDP-43 loss, axon maintenance
    'ATXN2',    # Ataxin-2, risk modifier, ASO therapeutic target (in clinical trials)
    'UNC13A',   # Cryptic exon target of TDP-43, recently identified as key mediator
    
    # From Pineda et al. (this dataset)
    'POU3F1',   # TF enriched in Betz/VEN neurons, altered localization + TDP-43 co-aggregation
    'SCN4B',    # Marks vulnerable L3/L5 projecting neurons
    
    # Cellular stress / vulnerability pathways
    'OPTN',     # Optineurin, autophagy receptor, ALS-linked
    'TBK1',     # TANK-binding kinase 1, innate immunity + autophagy
    'NEK1',     # DNA damage response, risk gene from GWAS
]

# Check which genes are in the dataset
available_genes = [g for g in ALS_GENES_CANDIDATES if g in adata.var_names]
missing_genes = [g for g in ALS_GENES_CANDIDATES if g not in adata.var_names]

print(f"Available ALS genes ({len(available_genes)}/{len(ALS_GENES_CANDIDATES)}):")
for gene in available_genes:
    gene_idx = list(adata.var_names).index(gene)
    if sparse.issparse(adata.X):
        expr = adata.X[:, gene_idx].toarray().flatten()
    else:
        expr = adata.X[:, gene_idx].flatten()
    pct_nonzero = np.mean(expr > 0) * 100
    print(f"  {gene:10s}  mean={expr.mean():.2f}  nonzero={pct_nonzero:.1f}%")

if missing_genes:
    print(f"\nMissing genes: {missing_genes}")

ALS_GENES = available_genes
print(f"\nUsing {len(ALS_GENES)} genes for perturbation")

In [ ]:
# Check expression of ALS genes in healthy vs disease cells
if DISEASE_COL:
    expr_comparison = []
    for gene in ALS_GENES:
        gene_idx = list(adata.var_names).index(gene)
        if sparse.issparse(adata.X):
            expr_all = adata.X[:, gene_idx].toarray().flatten()
        else:
            expr_all = adata.X[:, gene_idx].flatten()
        
        expr_comparison.append({
            'gene': gene,
            'mean_healthy': expr_all[healthy_mask].mean(),
            'mean_disease': expr_all[disease_mask].mean(),
            'pct_nz_healthy': np.mean(expr_all[healthy_mask] > 0) * 100,
            'pct_nz_disease': np.mean(expr_all[disease_mask] > 0) * 100,
        })
    
    df_expr = pd.DataFrame(expr_comparison)
    df_expr['log2fc'] = np.log2((df_expr['mean_disease'] + 0.01) / (df_expr['mean_healthy'] + 0.01))
    print(df_expr.to_string(index=False))
    
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(df_expr))
    width = 0.35
    ax.bar(x - width/2, df_expr['mean_healthy'], width, label='Healthy', color='steelblue')
    ax.bar(x + width/2, df_expr['mean_disease'], width, label='Disease (sALS)', color='indianred')
    ax.set_xticks(x)
    ax.set_xticklabels(df_expr['gene'], rotation=45, ha='right')
    ax.set_ylabel('Mean expression (raw counts)')
    ax.set_title('ALS gene expression: Healthy vs Disease')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../slides/task2_gene_expression_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

### Tokenization rank diagnostic

GeneFormer tokenizes each cell by rank-ordering genes by expression and keeping only the top 4,096. Genes that rank below this cutoff are invisible to the model — perturbation has no effect. We check which of our target genes fall within the tokenization window.

In [ ]:
CONTEXT_WINDOW = 4096

rank_diagnostics = []
for gene in ALS_GENES:
    gene_idx = list(adata.var_names).index(gene)
    if sparse.issparse(adata.X):
        gene_expr = adata.X[:, gene_idx].toarray().flatten()
    else:
        gene_expr = adata.X[:, gene_idx].flatten()
    
    # For each cell, count how many genes have higher expression than this gene
    # Gene is in top-4096 if fewer than 4096 genes are more highly expressed
    n_cells = adata.shape[0]
    
    # Efficient: compute rank of this gene per cell using the full expression matrix
    # For large matrices, sample cells for speed
    n_sample = min(5000, n_cells)
    rng_rank = np.random.RandomState(42)
    sample_idx = rng_rank.choice(n_cells, n_sample, replace=False)
    
    if sparse.issparse(adata.X):
        X_sample = adata.X[sample_idx].toarray()
    else:
        X_sample = adata.X[sample_idx]
    
    gene_expr_sample = X_sample[:, gene_idx]
    # Rank = number of genes with strictly higher expression (0 = highest expressed)
    ranks = (X_sample > gene_expr_sample[:, None]).sum(axis=1)
    
    in_window = (ranks < CONTEXT_WINDOW).mean() * 100
    median_rank = np.median(ranks)
    
    rank_diagnostics.append({
        'gene': gene,
        'median_rank': int(median_rank),
        'in_top_4096_pct': round(in_window, 1),
        'mean_expr': gene_expr.mean(),
        'pct_nonzero': round(np.mean(gene_expr > 0) * 100, 1),
        'visible_to_geneformer': in_window > 50,
    })

df_ranks = pd.DataFrame(rank_diagnostics).sort_values('median_rank')
print("Gene tokenization rank diagnostic:")
print("=" * 85)
print(df_ranks.to_string(index=False))
print(f"\nGenes visible to GeneFormer (>50% cells in top-4096): "
      f"{df_ranks['visible_to_geneformer'].sum()}/{len(df_ranks)}")

invisible_genes = df_ranks[~df_ranks['visible_to_geneformer']]['gene'].tolist()
if invisible_genes:
    print(f"\nWARNING: {invisible_genes} are below the tokenization cutoff in most cells.")
    print("Perturbation effects for these genes will be minimal or zero.")
    print("This is a fundamental limitation of GeneFormer's rank-based tokenization.")

In [ ]:
# Visualize tokenization ranks
fig, ax = plt.subplots(figsize=(10, max(5, len(df_ranks) * 0.4)))
colors = ['seagreen' if v else 'indianred' for v in df_ranks['visible_to_geneformer']]
bars = ax.barh(df_ranks['gene'], df_ranks['median_rank'], color=colors, edgecolor='gray')
ax.axvline(x=CONTEXT_WINDOW, color='black', linewidth=2, linestyle='--', label=f'Tokenization cutoff ({CONTEXT_WINDOW})')
ax.set_xlabel('Median expression rank (lower = more highly expressed)', fontsize=12)
ax.set_title('ALS gene expression rank vs GeneFormer tokenization window', fontsize=13)
ax.legend()
ax.invert_yaxis()

for i, (_, row) in enumerate(df_ranks.iterrows()):
    ax.text(row['median_rank'] + 100, i, f"{row['in_top_4096_pct']:.0f}% visible", 
            va='center', fontsize=9, color='gray')

plt.tight_layout()
plt.savefig('../slides/task2_tokenization_ranks.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Initialize GeneFormer and import perturbation functions

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
gf_config = GeneformerConfig(
    model_name="gf-12L-95M-i4096",
    device=device,
    batch_size=32,
)
geneformer = Geneformer(configurer=gf_config)
print(f"GeneFormer V2 loaded on {device}")

In [ ]:
# Import perturbation functions from Task 1
# (Duplicated here for notebook independence — each notebook is self-contained)

DOSE_LEVELS = {
    'knockout':      0.0,
    'strong_down':   0.1,
    'moderate_down': 0.5,
    'mild_up':       1.5,
    'moderate_up':   2.0,
    'strong_up':     5.0,
    'extreme_up':   10.0,
}

def perturb_gene(adata, gene_name, factor):
    """Scale a single gene's expression by `factor` across all cells."""
    if gene_name not in adata.var_names:
        raise ValueError(f"Gene '{gene_name}' not in dataset")
    adata_pert = adata.copy()
    gene_idx = list(adata_pert.var_names).index(gene_name)
    if sparse.issparse(adata_pert.X):
        X = adata_pert.X.tocsc()
        col = X.getcol(gene_idx).toarray().flatten() * factor
        col = np.round(col).astype(X.dtype)
        X_lil = X.tolil()
        X_lil[:, gene_idx] = col.reshape(-1, 1)
        adata_pert.X = X_lil.tocsr()
    else:
        adata_pert.X[:, gene_idx] = np.round(adata_pert.X[:, gene_idx] * factor).astype(adata.X.dtype)
    return adata_pert


def compute_perturbation_effect(emb_baseline, emb_perturbed):
    """Cosine distance + shift vectors between baseline and perturbed embeddings."""
    norm_base = emb_baseline / (np.linalg.norm(emb_baseline, axis=1, keepdims=True) + 1e-8)
    norm_pert = emb_perturbed / (np.linalg.norm(emb_perturbed, axis=1, keepdims=True) + 1e-8)
    cosine_dist = 1 - np.sum(norm_base * norm_pert, axis=1)
    shift_vectors = emb_perturbed - emb_baseline
    shift_magnitude = np.linalg.norm(shift_vectors, axis=1)
    return {
        'cosine_distance': cosine_dist,
        'shift_magnitude': shift_magnitude,
        'shift_vectors': shift_vectors,
    }

## 4. Run perturbations

We perturb ALS genes in **diseased (sALS) cells** and **healthy (PN) cells**. For speed, use `scripts/run_perturbations_parallel.py` across 8 GPUs. If pre-computed results exist, we load them and skip directly to analysis.

In [ ]:
RESULTS_DISEASE_PATH = '../data/results_disease.pkl'
RESULTS_HEALTHY_PATH = '../data/results_healthy.pkl'
REVERSAL_SCORES_PATH = '../data/reversal_scores.csv'

RESULTS_CACHED = (os.path.exists(RESULTS_DISEASE_PATH) and 
                  os.path.exists(RESULTS_HEALTHY_PATH) and
                  os.path.exists(REVERSAL_SCORES_PATH))

if RESULTS_CACHED:
    print("Pre-computed perturbation results found — loading from cache")
    print("  (Generated by scripts/run_perturbations_parallel.py)")
    with open(RESULTS_DISEASE_PATH, 'rb') as f:
        results_disease = pickle.load(f)
    with open(RESULTS_HEALTHY_PATH, 'rb') as f:
        results_healthy = pickle.load(f)
    df_reversal = pd.read_csv(REVERSAL_SCORES_PATH)
    print(f"  Disease: {len(results_disease)} genes")
    print(f"  Healthy: {len(results_healthy)} genes")
    print(f"  Reversal scores: {len(df_reversal)} rows")
else:
    print("No cached results — will compute perturbations in this notebook")

# Disease cell subset (needed for analysis either way)
emb_disease_baseline = embeddings_baseline[disease_mask]
print(f"\nDisease cells: {disease_mask.sum():,}")
print(f"Healthy cells: {healthy_mask.sum():,}")

In [ ]:
# Skip if results are cached from parallel script
if RESULTS_CACHED:
    print("Skipping computation — using cached results. Jump to Section 7 for analysis.")
else:
    pass  # Continue with the cells below

# Run perturbations on ALL disease cells for each gene and dose
# If dataset is very large (>50K cells), subsample for speed
MAX_CELLS = 20000
adata_disease = adata[disease_mask].copy()
if adata_disease.shape[0] > MAX_CELLS:
    print(f"Subsampling {MAX_CELLS} disease cells for perturbation (from {adata_disease.shape[0]})")
    rng = np.random.RandomState(42)
    idx = rng.choice(adata_disease.shape[0], MAX_CELLS, replace=False)
    adata_disease_sub = adata_disease[idx].copy()
    emb_disease_sub = emb_disease_baseline[idx]
else:
    adata_disease_sub = adata_disease
    emb_disease_sub = emb_disease_baseline

print(f"Running perturbations on {adata_disease_sub.shape[0]:,} disease cells")
print(f"Genes: {len(ALS_GENES)}, Doses: {len(DOSE_LEVELS)}")
print(f"Total forward passes: {len(ALS_GENES) * len(DOSE_LEVELS)}")

In [ ]:
# Main perturbation loop for disease cells
results_disease = {}

for gene in tqdm(ALS_GENES, desc="Genes (disease)"):
    results_disease[gene] = {}
    for dose_name, factor in DOSE_LEVELS.items():
        adata_pert = perturb_gene(adata_disease_sub, gene, factor)
        dataset_pert = geneformer.process_data(adata_pert)
        emb_pert = geneformer.get_embeddings(dataset_pert)
        
        effect = compute_perturbation_effect(emb_disease_sub, emb_pert)
        effect['embeddings'] = emb_pert
        effect['factor'] = factor
        results_disease[gene][dose_name] = effect
        
        print(f"  {gene:10s} | {dose_name:15s} (x{factor:5.1f}): "
              f"cos_dist={effect['cosine_distance'].mean():.4f}  "
              f"shift={effect['shift_magnitude'].mean():.4f}")

In [ ]:
# Save disease perturbation results
# Strip large embedding arrays for pickle size, keep metrics
results_disease_slim = {}
for gene in results_disease:
    results_disease_slim[gene] = {}
    for dose_name, effect in results_disease[gene].items():
        results_disease_slim[gene][dose_name] = {
            'cosine_distance': effect['cosine_distance'],
            'shift_magnitude': effect['shift_magnitude'],
            'shift_vectors': effect['shift_vectors'],
            'factor': effect['factor'],
        }

with open('../data/results_disease.pkl', 'wb') as f:
    pickle.dump(results_disease_slim, f)
print("Disease results saved")

## 5. Run perturbations on healthy cells

Perturbing healthy cells provides a **control**: does knocking down TARDBP in a healthy cell create a disease-like shift? This tests whether GeneFormer captures the disease mechanism.

In [ ]:
adata_healthy = adata[healthy_mask].copy()
emb_healthy_baseline = embeddings_baseline[healthy_mask]

MAX_CELLS_HEALTHY = 10000
if adata_healthy.shape[0] > MAX_CELLS_HEALTHY:
    rng = np.random.RandomState(42)
    idx = rng.choice(adata_healthy.shape[0], MAX_CELLS_HEALTHY, replace=False)
    adata_healthy_sub = adata_healthy[idx].copy()
    emb_healthy_sub = emb_healthy_baseline[idx]
else:
    adata_healthy_sub = adata_healthy
    emb_healthy_sub = emb_healthy_baseline

print(f"Running perturbations on {adata_healthy_sub.shape[0]:,} healthy cells")

In [ ]:
# Perturbation loop for healthy cells
results_healthy = {}

for gene in tqdm(ALS_GENES, desc="Genes (healthy)"):
    results_healthy[gene] = {}
    for dose_name, factor in DOSE_LEVELS.items():
        adata_pert = perturb_gene(adata_healthy_sub, gene, factor)
        dataset_pert = geneformer.process_data(adata_pert)
        emb_pert = geneformer.get_embeddings(dataset_pert)
        
        effect = compute_perturbation_effect(emb_healthy_sub, emb_pert)
        effect['embeddings'] = emb_pert
        effect['factor'] = factor
        results_healthy[gene][dose_name] = effect
        
        print(f"  {gene:10s} | {dose_name:15s} (x{factor:5.1f}): "
              f"cos_dist={effect['cosine_distance'].mean():.4f}  "
              f"shift={effect['shift_magnitude'].mean():.4f}")

In [ ]:
# Save healthy perturbation results
results_healthy_slim = {}
for gene in results_healthy:
    results_healthy_slim[gene] = {}
    for dose_name, effect in results_healthy[gene].items():
        results_healthy_slim[gene][dose_name] = {
            'cosine_distance': effect['cosine_distance'],
            'shift_magnitude': effect['shift_magnitude'],
            'shift_vectors': effect['shift_vectors'],
            'factor': effect['factor'],
        }

with open('../data/results_healthy.pkl', 'wb') as f:
    pickle.dump(results_healthy_slim, f)
print("Healthy results saved")

## 6. Compute disease-to-healthy direction

For Task 3 and Task 4, we need the direction in embedding space that points from disease toward health. We compute this from the baseline embeddings.

In [ ]:
# Centroids
centroid_healthy = emb_healthy_baseline.mean(axis=0)
centroid_disease = emb_disease_baseline.mean(axis=0)

# Disease-to-healthy direction (unit vector)
disease_to_healthy = centroid_healthy - centroid_disease
disease_to_healthy_norm = disease_to_healthy / (np.linalg.norm(disease_to_healthy) + 1e-8)

print(f"Centroid distance (disease → healthy): {np.linalg.norm(disease_to_healthy):.4f}")
print(f"Cosine distance between centroids: {1 - np.dot(centroid_healthy, centroid_disease) / (np.linalg.norm(centroid_healthy) * np.linalg.norm(centroid_disease)):.4f}")

# Save for Task 3
np.save('../data/centroid_healthy.npy', centroid_healthy)
np.save('../data/centroid_disease.npy', centroid_disease)
np.save('../data/disease_to_healthy_direction.npy', disease_to_healthy_norm)

In [ ]:
# For each perturbation of disease cells, measure how much the shift
# projects onto the disease-to-healthy direction
# Positive = moving toward healthy, negative = moving away

reversal_scores = []

for gene in ALS_GENES:
    for dose_name, effect in results_disease[gene].items():
        # Project shift vectors onto disease→healthy direction
        projections = effect['shift_vectors'] @ disease_to_healthy_norm
        
        reversal_scores.append({
            'gene': gene,
            'dose': dose_name,
            'factor': effect['factor'],
            'mean_reversal': projections.mean(),
            'std_reversal': projections.std(),
            'pct_toward_healthy': (projections > 0).mean() * 100,
            'mean_cosine_dist': effect['cosine_distance'].mean(),
        })

df_reversal = pd.DataFrame(reversal_scores)
df_reversal.to_csv('../data/reversal_scores.csv', index=False)
print("Reversal scores computed and saved")
print(f"\nTop perturbations by disease reversal (knock-down only):")
df_down = df_reversal[df_reversal['factor'] < 1].sort_values('mean_reversal', ascending=False)
print(df_down[['gene', 'dose', 'mean_reversal', 'pct_toward_healthy']].head(20).to_string(index=False))

## 7. Summary of perturbation effects

In [ ]:
# Heatmap: mean cosine distance per gene × dose
pivot = df_reversal.pivot_table(index='gene', columns='dose', values='mean_cosine_dist')
dose_order = [d for d in DOSE_LEVELS.keys() if d in pivot.columns]
pivot = pivot[dose_order]

fig, ax = plt.subplots(figsize=(12, max(6, len(ALS_GENES) * 0.5)))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax)
ax.set_title('Perturbation effect (cosine distance) — Disease cells', fontsize=13)
ax.set_xlabel('Dose level')
ax.set_ylabel('Gene')
plt.tight_layout()
plt.savefig('../slides/task2_perturbation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Dose-response curves for all genes
fig, ax = plt.subplots(figsize=(10, 6))
for gene in ALS_GENES:
    gene_data = df_reversal[df_reversal['gene'] == gene].sort_values('factor')
    ax.plot(gene_data['factor'], gene_data['mean_cosine_dist'], 'o-', label=gene, alpha=0.8)

ax.set_xscale('log')
ax.axvline(x=1.0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Expression scaling factor', fontsize=12)
ax.set_ylabel('Mean cosine distance', fontsize=12)
ax.set_title('Dose-response curves for ALS gene perturbations (disease cells)', fontsize=13)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('../slides/task2_dose_response_all.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Disease reversal: which knockdowns push disease cells toward healthy?
pivot_reversal = df_reversal.pivot_table(index='gene', columns='dose', values='mean_reversal')
pivot_reversal = pivot_reversal[dose_order]

fig, ax = plt.subplots(figsize=(12, max(6, len(ALS_GENES) * 0.5)))
sns.heatmap(pivot_reversal, annot=True, fmt='.4f', cmap='RdBu', center=0, ax=ax)
ax.set_title('Disease reversal score (positive = toward healthy) — Disease cells', fontsize=13)
ax.set_xlabel('Dose level')
ax.set_ylabel('Gene')
plt.tight_layout()
plt.savefig('../slides/task2_reversal_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

**Perturbations applied:**
- Genes: up to 12 ALS disease genes (canonical + dataset-specific)
- Doses: 7 levels per gene (knockout → 10x upregulation)
- Cell populations: disease (sALS) and healthy (control) cells separately

**Key outputs for Task 3:**
- `results_disease.pkl` / `results_healthy.pkl`: full perturbation metrics per gene × dose
- `reversal_scores.csv`: disease-to-healthy projection for each perturbation
- `centroid_healthy.npy` / `centroid_disease.npy`: embedding space anchors
- `disease_to_healthy_direction.npy`: unit vector for measuring therapeutic effect

**Initial observations:**
- Which genes show the strongest embedding shifts? (high sensitivity = important regulatory role)
- Which knockdowns move disease cells toward healthy? (positive reversal = therapeutic candidates)
- Are dose-response curves monotonic? (expected for direct regulators)